In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from datetime import date, timedelta

PROJECT_ROOT = Path.home() / "wspr-propagation"
WSPR_DATA_DIR = PROJECT_ROOT / "data" / "wspr"
SW_DATA_DIR   = PROJECT_ROOT / "data" / "spaceweather"

# Band codes
BANDS = [10, 20, 40]
BAND_CODES = {10: 28, 20: 14, 40: 7}

# Fetch schedule
def third_monday(year, month):
    d = date(year, month, 1)
    days_until_monday = (7 - d.weekday()) % 7
    return d + timedelta(days=days_until_monday) + timedelta(weeks=2)

fetch_schedule = []
for month in range(1, 13):
    start = third_monday(2023, month)
    for offset in range(7):
        fetch_schedule.append(start + timedelta(days=offset))

# Load space weather
df_sw = pd.read_parquet(SW_DATA_DIR / "spaceweather_2023.parquet")
df_flares = pd.read_parquet(SW_DATA_DIR / "flares_2023.parquet")

print(f"Space weather: {len(df_sw):,} rows")
print(f"Flares: {len(df_flares):,} events")
print(f"WSPR schedule: {len(fetch_schedule)} days × 3 bands = {len(fetch_schedule)*3} files")

In [ ]:
def load_wspr_band(band_m: int, schedule: list) -> pd.DataFrame:
    """
    Load all geo-enriched WSPR parquets for a given band.
    Joins space weather indices by nearest 3-hour Kp window.
    """
    frames = []
    missing = 0
    
    for d in schedule:
        date_str = d.strftime("%Y-%m-%d")
        path = WSPR_DATA_DIR / f"wspr_{date_str}_{band_m}m_geo.parquet"
        if path.exists():
            frames.append(pd.read_parquet(path))
        else:
            missing += 1
    
    if not frames:
        print(f"No data for {band_m}m")
        return pd.DataFrame()
    
    df = pd.concat(frames, ignore_index=True)
    print(f"{band_m}m: {len(df):,} rows from {len(frames)} days ({missing} missing)")
    
    # Join space weather — merge on nearest 3-hour Kp timestamp
    # Floor time to nearest 3 hours to match Kp cadence
    df["time_3h"] = df["time"].dt.floor("3h")
    df_sw_clean = df_sw[["time", "Kp", "ap", "Fobs", "SN"]].copy()
    df_sw_clean["time"] = pd.to_datetime(df_sw_clean["time"])
    df_sw_clean = df_sw_clean.rename(columns={"time": "time_3h"})
    
    df = df.merge(df_sw_clean, on="time_3h", how="left")
    df = df.drop(columns=["time_3h"])
    
    df["band_m"] = band_m
    return df

# Load all three bands
print("Loading WSPR data...")
df_10 = load_wspr_band(10, fetch_schedule)
df_20 = load_wspr_band(20, fetch_schedule)
df_40 = load_wspr_band(40, fetch_schedule)

df_all = pd.concat([df_10, df_20, df_40], ignore_index=True)
print(f"\nTotal rows: {len(df_all):,}")
print(f"Columns: {list(df_all.columns)}")

In [ ]:
del df_10, df_20, df_40, df_all
import gc
gc.collect()
print("Memory freed.")

In [ ]:
def load_wspr_aggregated(band_m: int, schedule: list, 
                          time_bin: str = "1h") -> pd.DataFrame:
    """
    Load WSPR data aggregated — never materializes full row-level DataFrame.
    For each file, computes summary stats per time bin × distance bucket × path_type,
    then concatenates the small summary frames.
    
    Returns a compact DataFrame suitable for interactive exploration.
    """
    frames = []
    missing = 0
    
    # Distance buckets in km
    dist_bins  = [0, 500, 1000, 2000, 4000, 8000, 12000]
    dist_labels = ["<500", "500-1k", "1k-2k", "2k-4k", "4k-8k", "8k+"]
    
    for d in schedule:
        date_str = d.strftime("%Y-%m-%d")
        path = WSPR_DATA_DIR / f"wspr_{date_str}_{band_m}m_geo.parquet"
        
        if not path.exists():
            missing += 1
            continue
        
        df = pd.read_parquet(path)
        
        # Join space weather
        df["time_3h"] = df["time"].dt.floor("3h")
        df_sw_clean = df_sw[["time","Kp","ap","Fobs","SN"]].copy()
        df_sw_clean["time"] = pd.to_datetime(df_sw_clean["time"])
        df_sw_clean = df_sw_clean.rename(columns={"time": "time_3h"})
        df = df.merge(df_sw_clean, on="time_3h", how="left")
        
        # Time bin
        df["time_bin"] = df["time"].dt.floor(time_bin)
        
        # Distance bucket
        df["dist_bucket"] = pd.cut(df["distance"], 
                                    bins=dist_bins, 
                                    labels=dist_labels)
        
        # Aggregate
        agg = df.groupby(
            ["time_bin", "dist_bucket", "path_type"],
            observed=True
        ).agg(
            spot_count    = ("snr",            "count"),
            snr_median    = ("snr",            "median"),
            snr_p25       = ("snr",            lambda x: x.quantile(0.25)),
            snr_p75       = ("snr",            lambda x: x.quantile(0.75)),
            path_loss_med = ("path_loss_proxy","median"),
            power_median  = ("power",          "median"),
            distance_med  = ("distance",       "median"),
            mid_sza_med   = ("mid_sza",        "median"),
            mid_mlat_med  = ("mid_mlat",       "median"),
            Kp_mean       = ("Kp",             "mean"),
            Fobs_mean     = ("Fobs",           "mean"),
            SN_mean       = ("SN",             "mean"),
        ).reset_index()
        
        agg["date"]   = date_str
        agg["band_m"] = band_m
        frames.append(agg)
        
        del df
    
    if not frames:
        return pd.DataFrame()
    
    result = pd.concat(frames, ignore_index=True)
    print(f"{band_m}m: {len(result):,} aggregate rows from {len(frames)} days ({missing} missing)")
    return result

print("Aggregating loader defined.")

In [ ]:
print("Loading aggregated data...")
agg_10 = load_wspr_aggregated(10, fetch_schedule)
agg_20 = load_wspr_aggregated(20, fetch_schedule)
agg_40 = load_wspr_aggregated(40, fetch_schedule)

df_agg = pd.concat([agg_10, agg_20, agg_40], ignore_index=True)

del agg_10, agg_20, agg_40
import gc
gc.collect()

print(f"\nAggregated DataFrame: {len(df_agg):,} rows")
print(f"Memory: {df_agg.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
print(df_agg.dtypes)
print(f"\nDate range: {df_agg['date'].min()} → {df_agg['date'].max()}")
print(f"Bands: {sorted(df_agg['band_m'].unique())}")
print(f"Path types: {df_agg['path_type'].unique()}")
print(f"Distance buckets: {df_agg['dist_bucket'].unique()}")
print(f"\nSpot count range: {df_agg['spot_count'].min()} to {df_agg['spot_count'].max():,}")
print(f"SNR median range: {df_agg['snr_median'].min():.1f} to {df_agg['snr_median'].max():.1f} dB")
print(f"Kp range: {df_agg['Kp_mean'].min():.1f} to {df_agg['Kp_mean'].max():.1f}")
print(f"Fobs range: {df_agg['Fobs_mean'].min():.1f} to {df_agg['Fobs_mean'].max():.1f}")
display(df_agg.head())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

band_colors = {10: "tomato", 20: "steelblue", 40: "seagreen"}
band_labels = {10: "10m (28 MHz)", 20: "20m (14 MHz)", 40: "40m (7 MHz)"}

for ax, band_m in zip(axes, [10, 20, 40]):
    df_band = df_agg[df_agg["band_m"] == band_m].copy()
    df_band["hour"] = pd.to_datetime(df_band["time_bin"]).dt.hour
    
    # Median SNR by hour and path type
    for path_type, ls in [("day", "-"), ("night", "--"), ("mixed", ":")]:
        subset = df_band[df_band["path_type"] == path_type]
        if subset.empty:
            continue
        hourly = subset.groupby("hour")["snr_median"].median()
        ax.plot(hourly.index, hourly.values, 
                color=band_colors[band_m], linestyle=ls,
                linewidth=1.5, label=path_type)
    
    ax.set_ylabel("Median SNR (dB)")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 5)

axes[-1].set_xlabel("Hour UTC")
axes[-1].set_xticks(range(0, 24, 2))
plt.suptitle("Median SNR by Hour UTC — All 2023 Data, NA+Europe", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

dist_colors = {
    "<500":   "purple",
    "500-1k": "royalblue", 
    "1k-2k":  "seagreen",
    "2k-4k":  "goldenrod",
    "4k-8k":  "tomato",
    "8k+":    "crimson"
}

for ax, band_m in zip(axes, [10, 20, 40]):
    df_band = df_agg[
        (df_agg["band_m"] == band_m) &
        (df_agg["path_type"] == "day")
    ].copy()
    df_band["hour"] = pd.to_datetime(df_band["time_bin"]).dt.hour
    
    for dist_bucket, color in dist_colors.items():
        subset = df_band[df_band["dist_bucket"] == dist_bucket]
        if len(subset) < 10:
            continue
        hourly = subset.groupby("hour")["snr_median"].median()
        ax.plot(hourly.index, hourly.values,
                color=color, linewidth=1.5, label=dist_bucket)
    
    ax.set_ylabel("Median SNR (dB)")
    ax.set_title(f"{band_m}m — Day paths only")
    ax.legend(loc="upper right", fontsize=8, title="Distance")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 5)

axes[-1].set_xlabel("Hour UTC")
axes[-1].set_xticks(range(0, 24, 2))
plt.suptitle("Median SNR by Hour UTC and Distance — Day Paths, 2023", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, band_m in zip(axes, [10, 20, 40]):
    df_band = df_agg[
        (df_agg["band_m"] == band_m) &
        (df_agg["path_type"] == "day")
    ].copy()
    
    # Bin SZA into 5-degree steps
    df_band["sza_bin"] = (df_band["mid_sza_med"] // 5) * 5
    
    for dist_bucket, color in dist_colors.items():
        subset = df_band[df_band["dist_bucket"] == dist_bucket]
        if len(subset) < 10:
            continue
        sza_curve = subset.groupby("sza_bin")["snr_median"].median()
        ax.plot(sza_curve.index, sza_curve.values,
                color=color, linewidth=1.5, label=dist_bucket)
    
    ax.set_ylabel("Median SNR (dB)")
    ax.set_title(f"{band_m}m — Day paths only")
    ax.legend(loc="upper right", fontsize=8, title="Distance")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 5)
    ax.invert_xaxis()  # 0° = overhead sun on left, 90° = horizon on right

axes[-1].set_xlabel("Solar Zenith Angle at Path Midpoint (°)\n← Sun overhead     Sun at horizon →")
axes[-1].set_xticks(range(0, 95, 5))
plt.suptitle("Median SNR vs Solar Zenith Angle — Day Paths, 2023", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Stratify 10m by SFI level
df_10m = df_agg[
    (df_agg["band_m"] == 10) &
    (df_agg["path_type"] == "day") &
    (df_agg["dist_bucket"].isin(["2k-4k", "4k-8k"]))  # DX distances where 10m shines
].copy()

# SFI bins
df_10m["sfi_bin"] = pd.cut(df_10m["Fobs_mean"],
                            bins=[100, 150, 200, 250, 350],
                            labels=["Low (100-150)", "Med (150-200)", 
                                    "High (200-250)", "Very High (250+)"])

df_10m["sza_bin"] = (df_10m["mid_sza_med"] // 5) * 5

fig, ax = plt.subplots(figsize=(14, 5))

sfi_colors = {
    "Low (100-150)":    "steelblue",
    "Med (150-200)":    "seagreen", 
    "High (200-250)":   "goldenrod",
    "Very High (250+)": "tomato"
}

for sfi_label, color in sfi_colors.items():
    subset = df_10m[df_10m["sfi_bin"] == sfi_label]
    if len(subset) < 5:
        continue
    curve = subset.groupby("sza_bin")["snr_median"].median()
    ax.plot(curve.index, curve.values,
            color=color, linewidth=2, label=f"SFI {sfi_label}")

ax.invert_xaxis()
ax.set_xlabel("Solar Zenith Angle (°)\n← Sun overhead     Sun at horizon →")
ax.set_ylabel("Median SNR (dB)")
ax.set_title("10m — SNR vs SZA stratified by Solar Flux (F10.7)\nDX paths 2k-8k km, Day only, 2023")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(-30, 5)
plt.tight_layout()
plt.show()

In [ ]:
# Check sample counts per SFI bin and SZA bin
counts = df_10m.groupby(["sfi_bin", "sza_bin"])["spot_count"].sum().unstack("sfi_bin")
print("Spot counts per SFI bin and SZA bin:")
print(counts.to_string())

In [ ]:
df_20m = df_agg[
    (df_agg["band_m"] == 20) &
    (df_agg["dist_bucket"].isin(["2k-4k", "4k-8k"]))
].copy()

df_20m["kp_bin"] = pd.cut(df_20m["Kp_mean"],
                           bins=[0, 2, 4, 6, 9],
                           labels=["Quiet (0-2)", "Unsettled (2-4)", 
                                   "Storm (4-6)", "Severe (6+)"])
df_20m["sza_bin"] = (df_20m["mid_sza_med"] // 5) * 5

# Check counts first
counts_kp = df_20m.groupby(["kp_bin","path_type"])["spot_count"].sum().unstack("path_type")
print("20m DX spot counts by Kp bin and path type:")
print(counts_kp.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

kp_colors = {
    "Quiet (0-2)":      "steelblue",
    "Unsettled (2-4)":  "seagreen",
    "Storm (4-6)":      "goldenrod",
    "Severe (6+)":      "tomato"
}

path_types = ["day", "mixed", "night"]

for ax, path_type in zip(axes, path_types):
    for kp_label, color in kp_colors.items():
        subset = df_20m[
            (df_20m["kp_bin"] == kp_label) &
            (df_20m["path_type"] == path_type)
        ]
        if len(subset) < 5:
            continue
        curve = subset.groupby("sza_bin")["snr_median"].median()
        ax.plot(curve.index, curve.values,
                color=color, linewidth=2, label=kp_label)
    
    ax.invert_xaxis()
    ax.set_xlabel("Solar Zenith Angle (°)\n← Overhead     Horizon →")
    ax.set_title(f"{path_type.capitalize()} paths")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 0)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Median SNR (dB)")
plt.suptitle("20m DX (2k-8k km) — Kp Effect on SNR by Path Type, 2023", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
df_20m_kp = df_agg[
    (df_agg["band_m"] == 20) &
    (df_agg["dist_bucket"].isin(["2k-4k", "4k-8k"])) &
    (df_agg["path_type"] == "day")
].copy()

df_20m_kp["kp_bin"] = pd.cut(df_20m_kp["Kp_mean"],
                               bins=[0, 2, 4, 6, 9],
                               labels=["Quiet (0-2)", "Unsettled (2-4)",
                                       "Storm (4-6)", "Severe (6+)"])

# Split by magnetic latitude of path midpoint
df_20m_kp["mlat_bin"] = pd.cut(df_20m_kp["mid_mlat_med"],
                                 bins=[0, 40, 55, 90],
                                 labels=["Low (<40°)", "Mid (40-55°)", "High (55°+)"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, mlat_label in zip(axes, ["Low (<40°)", "Mid (40-55°)", "High (55°+)"]):
    subset_mlat = df_20m_kp[df_20m_kp["mlat_bin"] == mlat_label]
    
    for kp_label, color in kp_colors.items():
        subset = subset_mlat[subset_mlat["kp_bin"] == kp_label]
        if len(subset) < 5:
            continue
        curve = subset.groupby(
            (subset["mid_sza_med"] // 5) * 5
        )["snr_median"].median()
        ax.plot(curve.index, curve.values,
                color=color, linewidth=2, label=kp_label)
    
    # Show sample size
    counts = subset_mlat.groupby("kp_bin")["spot_count"].sum()
    count_str = "\n".join([f"{k}: {v:,.0f}" for k,v in counts.items()])
    ax.text(0.02, 0.02, count_str, transform=ax.transAxes,
            fontsize=7, verticalalignment='bottom', family='monospace')
    
    ax.invert_xaxis()
    ax.set_xlabel("Solar Zenith Angle (°)\n← Overhead     Horizon →")
    ax.set_title(f"Mag. lat {mlat_label}")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 0)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Median SNR (dB)")
plt.suptitle("20m Day DX — Kp Effect by Magnetic Latitude of Path Midpoint, 2023", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
df_seasonal = df_agg[
    (df_agg["band_m"].isin([10, 20, 40])) &
    (df_agg["path_type"] == "day") &
    (df_agg["dist_bucket"].isin(["2k-4k", "4k-8k"]))
].copy()

df_seasonal["month"] = pd.to_datetime(df_seasonal["date"]).dt.month
df_seasonal["sza_bin"] = (df_seasonal["mid_sza_med"] // 10) * 10

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

month_colors = plt.cm.hsv(np.linspace(0, 0.85, 12))

for ax, band_m in zip(axes, [10, 20, 40]):
    subset_band = df_seasonal[df_seasonal["band_m"] == band_m]
    
    for month in range(1, 13):
        subset = subset_band[subset_band["month"] == month]
        if len(subset) < 5:
            continue
        curve = subset.groupby("sza_bin")["snr_median"].median()
        ax.plot(curve.index, curve.values,
                color=month_colors[month-1],
                linewidth=1.5,
                label=f"Month {month:02d}")
    
    ax.invert_xaxis()
    ax.set_xlabel("Solar Zenith Angle (°)\n← Overhead     Horizon →")
    ax.set_title(f"{band_m}m")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 0)
    ax.legend(fontsize=7, ncol=2, title="Month")

axes[0].set_ylabel("Median SNR (dB)")
plt.suptitle("Seasonal Variation — Day DX Paths 2k-8k km, 2023", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Save session state
df_agg.to_parquet(PROJECT_ROOT / "data" / "explore_agg_v1.parquet", index=False)
print("Session state saved.")

In [ ]:
# Load flares
df_flares = pd.read_parquet(SW_DATA_DIR / "flares_2023.parquet")

# Filter to M and X class only
df_mx = df_flares[df_flares["flare_class"].str[0].isin(["M", "X"])].copy()
df_mx["date"] = pd.to_datetime(df_mx["peak"]).dt.date.astype(str)

# Which flares fall on days we have WSPR data?
wspr_dates = set(d.strftime("%Y-%m-%d") for d in fetch_schedule)
df_mx["have_wspr"] = df_mx["date"].isin(wspr_dates)

print(f"Total M+X flares in 2023: {len(df_mx)}")
print(f"Flares on WSPR days: {df_mx['have_wspr'].sum()}")
print(f"\nBy class:")
print(df_mx.groupby([df_mx["flare_class"].str[0], "have_wspr"]).size().unstack("have_wspr"))
print(f"\nX-class flares on WSPR days:")
print(df_mx[df_mx["flare_class"].str[0]=="X"][["peak","flare_class","have_wspr"]])

In [ ]:
# Load June 20 data for all three bands at full hourly resolution
# X1.1 flare peaked at 17:09 UTC

flare_date = "2023-06-19"  # June 19 is the Monday of that week — check
# Actually June 20 is a Tuesday, let's verify it's in our schedule
print("June 2023 schedule dates:")
for d in fetch_schedule:
    if d.month == 6:
        print(f"  {d.strftime('%Y-%m-%d')} ({d.strftime('%A')})")

In [ ]:
# Load June 20 raw geo data for case study
frames = []
for band_m in [10, 20, 40]:
    path = WSPR_DATA_DIR / f"wspr_2023-06-20_{band_m}m_geo.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        df["band_m"] = band_m
        frames.append(df)
        print(f"{band_m}m: {len(df):,} rows")

df_june20 = pd.concat(frames, ignore_index=True)

# Add space weather
df_june20["time_3h"] = df_june20["time"].dt.floor("3h")
df_sw_clean = df_sw[["time","Kp","ap","Fobs","SN"]].copy()
df_sw_clean["time"] = pd.to_datetime(df_sw_clean["time"])
df_sw_clean = df_sw_clean.rename(columns={"time": "time_3h"})
df_june20 = df_june20.merge(df_sw_clean, on="time_3h", how="left")

# Flare details
flare = df_mx[df_mx["flare_class"].str[0] == "X"].iloc[
    df_mx[df_mx["flare_class"].str[0] == "X"]["have_wspr"].values.tolist().index(True)
]
print(f"\nFlare: {flare['flare_class']} peak at {flare['peak']}")
print(f"Begin: {flare['begin']}, End: {flare['end']}")
print(f"Peak flux: {flare['peak_flux']:.4f} W/m²")
print(f"\nTotal rows loaded: {len(df_june20):,}")

In [ ]:
# Add dist_bucket to df_june20
dist_bins   = [0, 500, 1000, 2000, 4000, 8000, 12000]
dist_labels = ["<500", "500-1k", "1k-2k", "2k-4k", "4k-8k", "8k+"]

df_june20["dist_bucket"] = pd.cut(df_june20["distance"],
                                   bins=dist_bins,
                                   labels=dist_labels)

print(f"dist_bucket added: {df_june20['dist_bucket'].value_counts().sort_index().to_dict()}")

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

flare_begin = pd.Timestamp("2023-06-20 16:42:00")
flare_peak  = pd.Timestamp("2023-06-20 17:09:00")
flare_end   = pd.Timestamp("2023-06-20 17:26:00")

# Aggregate to 30-minute bins for finer resolution than df_agg
df_june20["time_30m"] = df_june20["time"].dt.floor("30min")
df_june20["hour_f"] = df_june20["time_30m"].dt.hour + df_june20["time_30m"].dt.minute / 60

# Only day paths with midpoint on sunlit hemisphere — SID only affects dayside
df_day = df_june20[
    (df_june20["path_type"] == "day") &
    (df_june20["mid_sza"] < 90) &
    (df_june20["dist_bucket"].isin(["1k-2k", "2k-4k", "4k-8k"]))
].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

band_colors = {10: "tomato", 20: "steelblue", 40: "seagreen"}
band_labels = {10: "10m (28 MHz)", 20: "20m (14 MHz)", 40: "40m (7 MHz)"}

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_day[df_day["band_m"] == band_m]
    
    if subset.empty:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center")
        continue
    
    # Median SNR per 30-min bin
    hourly = subset.groupby("time_30m")["snr"].agg(["median", "count"])
    hourly = hourly[hourly["count"] >= 10]  # require at least 10 spots
    
    hour_vals = hourly.index.hour + hourly.index.minute / 60
    
    ax.plot(hour_vals, hourly["median"],
            color=band_colors[band_m], linewidth=2, zorder=3)
    
    # Flare window shading
    ax.axvspan(flare_begin.hour + flare_begin.minute/60,
               flare_end.hour + flare_end.minute/60,
               alpha=0.15, color="red", zorder=1, label="Flare window")
    ax.axvline(flare_peak.hour + flare_peak.minute/60,
               color="red", linewidth=1.5, linestyle="--", zorder=2, label="Flare peak")
    
    # Compute pre/post flare baseline
    pre_mask  = (hour_vals >= 14) & (hour_vals < 16.5)
    post_mask = (hour_vals >= 17.5) & (hour_vals < 20)
    flare_mask = (hour_vals >= 16.5) & (hour_vals <= 17.5)
    
    if pre_mask.any():
        pre_snr = hourly["median"][pre_mask.values].mean()
        ax.axhline(pre_snr, color=band_colors[band_m], linewidth=1,
                   linestyle=":", alpha=0.6, label=f"Pre-flare baseline: {pre_snr:.1f} dB")
    
    ax.set_ylabel("Median SNR (dB)")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 5)

axes[-1].set_xlabel("Hour UTC")
axes[-1].set_xticks(range(0, 24, 1))
plt.suptitle(f"X1.1 Solar Flare Case Study — June 20, 2023\n"
             f"Flare: 16:42 → 17:09 peak → 17:26 UTC | Day paths 1k-8k km, sunlit midpoint",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("path_type values:", df_june20["path_type"].unique())
print("path_type dtype:", df_june20["path_type"].dtype)
print("\nBand counts before filter:")
print(df_june20.groupby("band_m")["path_type"].value_counts())
print("\nAfter day filter:")
df_test = df_june20[df_june20["path_type"] == "day"]
print(df_test.groupby("band_m").size())
print("\nmid_sza range:", df_june20["mid_sza"].min(), "to", df_june20["mid_sza"].max())

In [ ]:
df_day_test = df_june20[
    (df_june20["path_type"] == "day") &
    (df_june20["mid_sza"] < 90)
].copy()

print("Rows per band after day+sza filter:")
print(df_day_test.groupby("band_m").size())

print("\nDist bucket counts per band:")
print(df_day_test.groupby(["band_m", "dist_bucket"]).size().unstack("dist_bucket"))

In [ ]:
flare_begin = pd.Timestamp("2023-06-20 16:42:00")
flare_peak  = pd.Timestamp("2023-06-20 17:09:00")
flare_end   = pd.Timestamp("2023-06-20 17:26:00")

df_june20["time_30m"] = df_june20["time"].dt.floor("30min")

# Band-appropriate distance filters
band_dist = {
    10: ["1k-2k"],
    20: ["1k-2k", "2k-4k"],
    40: ["<500", "500-1k"]
}

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
band_colors = {10: "tomato", 20: "steelblue", 40: "seagreen"}
band_labels = {10: "10m (28 MHz)", 20: "20m (14 MHz)", 40: "40m (7 MHz)"}

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_june20[
        (df_june20["band_m"] == band_m) &
        (df_june20["path_type"] == "day") &
        (df_june20["mid_sza"] < 90) &
        (df_june20["dist_bucket"].isin(band_dist[band_m]))
    ].copy()

    if subset.empty:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center")
        continue

    hourly = subset.groupby("time_30m")["snr"].agg(["median", "count"])
    hourly = hourly[hourly["count"] >= 10]

    hour_vals = hourly.index.hour + hourly.index.minute / 60

    ax.plot(hour_vals, hourly["median"],
            color=band_colors[band_m], linewidth=2, zorder=3, label="Median SNR")

    # Flare window
    ax.axvspan(flare_begin.hour + flare_begin.minute/60,
               flare_end.hour + flare_end.minute/60,
               alpha=0.15, color="red", zorder=1)
    ax.axvline(flare_peak.hour + flare_peak.minute/60,
               color="red", linewidth=1.5, linestyle="--", zorder=2, label="X1.1 peak 17:09")

    # Pre-flare baseline (14:00-16:30)
    pre_mask = (hour_vals >= 14) & (hour_vals < 16.5)
    if pre_mask.any():
        pre_snr = hourly["median"][pre_mask].mean()
        ax.axhline(pre_snr, color=band_colors[band_m], linewidth=1,
                   linestyle=":", alpha=0.7, label=f"Pre-flare baseline: {pre_snr:.1f} dB")

    dist_str = ", ".join(band_dist[band_m])
    ax.set_ylabel("Median SNR (dB)")
    ax.set_title(f"{band_labels[band_m]} — day paths, distance: {dist_str}")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-30, 5)

axes[-1].set_xlabel("Hour UTC")
axes[-1].set_xticks(range(0, 24, 1))
plt.suptitle(f"X1.1 Solar Flare Case Study — June 20, 2023\n"
             f"Flare begin 16:42 → peak 17:09 → end 17:26 UTC",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the flare effect for each band
print("Flare SID analysis — X1.1, June 20 2023")
print("=" * 55)

for band_m in [10, 20, 40]:
    subset = df_june20[
        (df_june20["band_m"] == band_m) &
        (df_june20["path_type"] == "day") &
        (df_june20["mid_sza"] < 90) &
        (df_june20["dist_bucket"].isin(band_dist[band_m]))
    ].copy()

    subset["time_30m"] = subset["time"].dt.floor("30min")
    hourly = subset.groupby("time_30m")["snr"].agg(["median","count"])
    hourly = hourly[hourly["count"] >= 10]
    hour_vals = hourly.index.hour + hourly.index.minute / 60

    pre_snr   = hourly["median"][(hour_vals >= 14) & (hour_vals < 16.5)].mean()
    flare_snr = hourly["median"][(hour_vals >= 16.5) & (hour_vals <= 17.5)].min()
    post_snr  = hourly["median"][(hour_vals > 17.5) & (hour_vals <= 19)].mean()

    depression = flare_snr - pre_snr
    recovery   = post_snr - flare_snr

    print(f"\n{band_m}m ({band_dist[band_m]}):")
    print(f"  Pre-flare baseline:  {pre_snr:.1f} dB")
    print(f"  Flare minimum SNR:   {flare_snr:.1f} dB")
    print(f"  Depression:          {depression:.1f} dB")
    print(f"  Post-flare recovery: {post_snr:.1f} dB (Δ {recovery:.1f} dB from min)")

In [ ]:
from datetime import timedelta

# Ensemble M-class flare analysis
# For each M-class flare on a WSPR day, extract a ±3 hour window
# normalized to pre-flare baseline

def load_day_band(date_str, band_m):
    path = WSPR_DATA_DIR / f"wspr_{date_str}_{band_m}m_geo.parquet"
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_parquet(path)
    df["band_m"] = band_m
    dist_bins   = [0, 500, 1000, 2000, 4000, 8000, 12000]
    dist_labels = ["<500", "500-1k", "1k-2k", "2k-4k", "4k-8k", "8k+"]
    df["dist_bucket"] = pd.cut(df["distance"], bins=dist_bins, labels=dist_labels)
    return df

# M-class flares on WSPR days
df_m = df_mx[
    (df_mx["flare_class"].str[0] == "M") &
    (df_mx["have_wspr"] == True)
].copy()
print(f"M-class flares to analyze: {len(df_m)}")

# For each band, collect normalized flare responses
WINDOW_HOURS = 3
BIN_MINUTES  = 30

results = {10: [], 20: [], 40: []}
band_dist_ensemble = {10: ["1k-2k"], 20: ["1k-2k","2k-4k"], 40: ["<500","500-1k"]}

for _, flare_row in df_m.iterrows():
    peak_time = pd.Timestamp(flare_row["peak"])
    date_str  = peak_time.strftime("%Y-%m-%d")
    
    for band_m in [10, 20, 40]:
        df_day = load_day_band(date_str, band_m)
        if df_day.empty:
            continue
        
        subset = df_day[
            (df_day["path_type"] == "day") &
            (df_day["mid_sza"] < 90) &
            (df_day["dist_bucket"].isin(band_dist_ensemble[band_m]))
        ].copy()
        
        if subset.empty:
            continue
        
        # 30-min bins in ±3 hour window
        subset["time_30m"] = subset["time"].dt.floor("30min")
        window_start = peak_time - pd.Timedelta(hours=WINDOW_HOURS)
        window_end   = peak_time + pd.Timedelta(hours=WINDOW_HOURS)
        
        subset = subset[
            (subset["time"] >= window_start) &
            (subset["time"] <= window_end)
        ]
        
        if subset.empty:
            continue
        
        hourly = subset.groupby("time_30m")["snr"].agg(["median","count"])
        hourly = hourly[hourly["count"] >= 5]
        
        if len(hourly) < 4:
            continue
        
        # Time offset in minutes from peak
        hourly["offset_min"] = (hourly.index - peak_time).total_seconds() / 60
        
        # Pre-flare baseline: -180 to -30 min before peak
        pre = hourly[(hourly["offset_min"] >= -180) & (hourly["offset_min"] <= -30)]
        if pre.empty:
            continue
        
        baseline = pre["median"].mean()
        hourly["snr_norm"] = hourly["median"] - baseline
        hourly["flare_class"] = flare_row["flare_class"]
        
        results[band_m].append(hourly[["offset_min","snr_norm","count","flare_class"]])

print("\nFlares contributing per band:")
for band_m in [10, 20, 40]:
    print(f"  {band_m}m: {len(results[band_m])} flares")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

band_colors = {10: "tomato", 20: "steelblue", 40: "seagreen"}
band_labels = {10: "10m (28 MHz)", 20: "20m (14 MHz)", 40: "40m (7 MHz)"}

for ax, band_m in zip(axes, [10, 20, 40]):
    if not results[band_m]:
        continue
    
    # Combine all flare windows
    df_ensemble = pd.concat(results[band_m], ignore_index=True)
    
    # Bin to 30-min offset bins
    df_ensemble["offset_bin"] = (df_ensemble["offset_min"] // 30) * 30
    
    grouped = df_ensemble.groupby("offset_bin")["snr_norm"].agg(
        ["median", "mean", "std", "count"]
    )
    
    # Only plot bins with at least 10 flare contributions
    grouped = grouped[grouped["count"] >= 10]
    
    x = grouped.index.values
    y = grouped["median"].values
    std = grouped["std"].values
    
    # Shaded uncertainty band
    ax.fill_between(x, y - std/2, y + std/2,
                    alpha=0.2, color=band_colors[band_m])
    ax.plot(x, y, color=band_colors[band_m], linewidth=2.5,
            label=f"Median (n={len(results[band_m])} flares)")
    
    # Reference lines
    ax.axhline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7)
    ax.axvline(0, color="red", linewidth=1.5, linestyle="--", alpha=0.8,
               label="Flare peak")
    ax.axvspan(-44, 17, alpha=0.08, color="red",
               label="Typical flare duration")
    
    ax.set_xlabel("Minutes relative to flare peak")
    ax.set_title(f"{band_labels[band_m]}\n{band_dist_ensemble[band_m]}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-6, 4)
    ax.set_xlim(-180, 180)

axes[0].set_ylabel("SNR anomaly relative to pre-flare baseline (dB)")
plt.suptitle("Ensemble M-class Flare Response — 2023, NA+Europe\n"
             "SNR normalized to pre-flare baseline, day paths only",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Stratify by flare subclass magnitude
for band_m in [10, 20, 40]:
    if not results[band_m]:
        continue
    df_ensemble = pd.concat(results[band_m], ignore_index=True)
    
    # Extract numeric magnitude from flare class e.g. M1.5 -> 1.5, M9.2 -> 9.2
    df_ensemble["magnitude"] = df_ensemble["flare_class"].str[1:].astype(float)
    df_ensemble["mag_bin"] = pd.cut(df_ensemble["magnitude"],
                                     bins=[0, 2, 5, 10],
                                     labels=["M1-2", "M2-5", "M5+"])
    results[band_m] = [df_ensemble]  # replace list with single combined df

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

mag_colors = {"M1-2": "steelblue", "M2-5": "goldenrod", "M5+": "tomato"}

for ax, band_m in zip(axes, [10, 20, 40]):
    df_e = results[band_m][0]
    df_e["offset_bin"] = (df_e["offset_min"] // 30) * 30
    
    for mag_label, color in mag_colors.items():
        subset = df_e[df_e["mag_bin"] == mag_label]
        if subset.empty:
            continue
        grouped = subset.groupby("offset_bin")["snr_norm"].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 5]
        if grouped.empty:
            continue
        n_flares = subset["flare_class"].nunique()
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, label=f"{mag_label} (n≈{n_flares})")
    
    ax.axhline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7)
    ax.axvline(0, color="red", linewidth=1.5, linestyle="--", alpha=0.8)
    ax.axvspan(-44, 17, alpha=0.08, color="red")
    
    ax.set_xlabel("Minutes relative to flare peak")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-6, 4)
    ax.set_xlim(-180, 180)

axes[0].set_ylabel("SNR anomaly (dB)")
plt.suptitle("M-class Flare Response by Magnitude — 2023, NA+Europe\n"
             "Day paths only, normalized to pre-flare baseline",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Rebuild ensemble cleanly from scratch
results_raw = {10: [], 20: [], 40: []}

df_m_fresh = df_mx[
    (df_mx["flare_class"].str[0] == "M") &
    (df_mx["have_wspr"] == True)
].copy()
df_m_fresh["magnitude"] = df_m_fresh["flare_class"].str[1:].astype(float)
df_m_fresh["mag_bin"] = pd.cut(df_m_fresh["magnitude"],
                                bins=[0, 2, 5, 10],
                                labels=["M1-2", "M2-5", "M5+"])

print(f"Flares by magnitude bin:")
print(df_m_fresh["mag_bin"].value_counts().sort_index())

In [ ]:
results_mag = {"M1-2": {10:[], 20:[], 40:[]},
               "M2-5": {10:[], 20:[], 40:[]},
               "M5+":  {10:[], 20:[], 40:[]}}

for _, flare_row in df_m_fresh.iterrows():
    peak_time = pd.Timestamp(flare_row["peak"])
    date_str  = peak_time.strftime("%Y-%m-%d")
    mag_bin   = flare_row["mag_bin"]

    for band_m in [10, 20, 40]:
        df_day = load_day_band(date_str, band_m)
        if df_day.empty:
            continue

        subset = df_day[
            (df_day["path_type"] == "day") &
            (df_day["mid_sza"] < 90) &
            (df_day["dist_bucket"].isin(band_dist_ensemble[band_m]))
        ].copy()

        if subset.empty:
            continue

        subset["time_30m"] = subset["time"].dt.floor("30min")
        window_start = peak_time - pd.Timedelta(hours=3)
        window_end   = peak_time + pd.Timedelta(hours=3)
        subset = subset[(subset["time"] >= window_start) & (subset["time"] <= window_end)]

        if subset.empty:
            continue

        hourly = subset.groupby("time_30m")["snr"].agg(["median","count"])
        hourly = hourly[hourly["count"] >= 5]

        if len(hourly) < 4:
            continue

        hourly["offset_min"] = (hourly.index - peak_time).total_seconds() / 60

        pre = hourly[(hourly["offset_min"] >= -180) & (hourly["offset_min"] <= -30)]
        if pre.empty:
            continue

        baseline = pre["median"].mean()
        hourly["snr_norm"] = hourly["median"] - baseline
        results_mag[mag_bin][band_m].append(hourly[["offset_min","snr_norm"]])

print("Flares contributing per magnitude bin and band:")
for mag in ["M1-2","M2-5","M5+"]:
    counts = {b: len(results_mag[mag][b]) for b in [10,20,40]}
    print(f"  {mag}: 10m={counts[10]}, 20m={counts[20]}, 40m={counts[40]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

mag_colors = {"M1-2": "steelblue", "M2-5": "goldenrod", "M5+": "tomato"}

for ax, band_m in zip(axes, [10, 20, 40]):
    for mag_bin, color in mag_colors.items():
        frames = results_mag[mag_bin][band_m]
        if not frames:
            continue
        
        df_e = pd.concat(frames, ignore_index=True)
        df_e["offset_bin"] = (df_e["offset_min"] // 30) * 30
        
        grouped = df_e.groupby("offset_bin")["snr_norm"].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 5]
        
        n = len(frames)
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, label=f"{mag_bin} (n={n})")
        ax.fill_between(grouped.index,
                        grouped["median"] - grouped["median"].std(),
                        grouped["median"] + grouped["median"].std(),
                        alpha=0.12, color=color)

    ax.axhline(0, color="gray", linewidth=1, linestyle="--", alpha=0.7)
    ax.axvline(0, color="red", linewidth=1.5, linestyle="--", alpha=0.8)
    ax.axvspan(-44, 17, alpha=0.08, color="red", label="Typical M flare duration")
    ax.set_xlabel("Minutes relative to flare peak")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-4, 4)
    ax.set_xlim(-180, 180)

axes[0].set_ylabel("SNR anomaly (dB)")
plt.suptitle("M-class Flare Response by Magnitude — 2023, NA+Europe\n"
             "Day paths only, normalized to pre-flare baseline",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("March and September schedule dates:")
for d in fetch_schedule:
    if d.month in [3, 9]:
        print(f"  {d.strftime('%Y-%m-%d')} ({d.strftime('%A')})")

In [ ]:
# Load equinox weeks at full resolution
equinox_dates = [d for d in fetch_schedule if d.month in [3, 9]]
print(f"Loading {len(equinox_dates)} days × 3 bands...")

equinox_frames = []
for d in equinox_dates:
    date_str = d.strftime("%Y-%m-%d")
    for band_m in [10, 20, 40]:
        path = WSPR_DATA_DIR / f"wspr_{date_str}_{band_m}m_geo.parquet"
        if not path.exists():
            continue
        df = pd.read_parquet(path)
        df["band_m"] = band_m
        df["date_str"] = date_str
        df["month"] = d.month
        df["season"] = "spring" if d.month == 3 else "autumn"
        equinox_frames.append(df)

df_eq = pd.concat(equinox_frames, ignore_index=True)

# Add distance bucket
dist_bins   = [0, 500, 1000, 2000, 4000, 8000, 12000]
dist_labels = ["<500", "500-1k", "1k-2k", "2k-4k", "4k-8k", "8k+"]
df_eq["dist_bucket"] = pd.cut(df_eq["distance"], bins=dist_bins, labels=dist_labels)

print(f"Total rows: {len(df_eq):,}")
print(f"Bands: {df_eq['band_m'].unique()}")
print(f"Seasons: {df_eq['season'].value_counts().to_dict()}")

In [ ]:
# Focus on the terminator zone — SZA 70° to 110° at path midpoint
# Use 15-minute time bins and 2° SZA bins for resolution

df_eq["time_15m"] = df_eq["time"].dt.floor("15min")
df_eq["sza_bin_2"] = (df_eq["mid_sza"] // 2) * 2

# Filter to terminator zone and DX distances
df_gl = df_eq[
    (df_eq["mid_sza"] >= 70) &
    (df_eq["mid_sza"] <= 110) &
    (df_eq["dist_bucket"].isin(["1k-2k", "2k-4k", "4k-8k"]))
].copy()

print(f"Terminator zone rows: {len(df_gl):,}")
print(f"\nRows per band:")
print(df_gl.groupby("band_m").size())
print(f"\nSZA distribution:")
print(df_gl.groupby(df_gl["sza_bin_2"].astype(int)).size().to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)

band_colors  = {10: "tomato", 20: "steelblue", 40: "seagreen"}
band_labels  = {10: "10m (28 MHz)", 20: "20m (14 MHz)", 40: "40m (7 MHz)"}
season_ls    = {"spring": "-", "autumn": "--"}
season_color_offset = {"spring": 0.0, "autumn": 0.3}

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_gl[df_gl["band_m"] == band_m]
    
    for season in ["spring", "autumn"]:
        ss = subset[subset["season"] == season]
        if ss.empty:
            continue
        
        grouped = ss.groupby("sza_bin_2")["snr"].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 50]
        
        ax.plot(grouped.index, grouped["median"],
                color=band_colors[band_m],
                linestyle=season_ls[season],
                linewidth=2.5,
                alpha=0.9 if season == "spring" else 0.6,
                label=f"{season.capitalize()}")
    
    # Mark the terminator
    ax.axvline(90, color="black", linewidth=1.5, linestyle=":",
               alpha=0.7, label="Terminator (SZA=90°)")
    ax.axvspan(85, 95, alpha=0.08, color="gold", label="Gray line zone")
    
    # Labels
    ax.set_xlabel("Solar Zenith Angle at Path Midpoint (°)\n← Daylight     Darkness →")
    ax.set_title(f"{band_labels[band_m]}\nDX paths 1k-8k km, equinox weeks")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Annotate daylight/night sides
    ax.text(0.05, 0.05, "← Daylight", transform=ax.transAxes,
            fontsize=8, color="goldenrod", alpha=0.8)
    ax.text(0.75, 0.05, "Darkness →", transform=ax.transAxes,
            fontsize=8, color="navy", alpha=0.8)

axes[0].set_ylabel("Median SNR (dB)")
plt.suptitle("Gray Line SNR Profile — Equinox Weeks 2023\n"
             "SNR vs Solar Zenith Angle at Path Midpoint, DX Paths",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom into SZA 82-98° with 1° bins
df_zoom = df_eq[
    (df_eq["mid_sza"] >= 82) &
    (df_eq["mid_sza"] <= 98) &
    (df_eq["dist_bucket"].isin(["1k-2k", "2k-4k", "4k-8k"]))
].copy()

df_zoom["sza_bin_1"] = (df_zoom["mid_sza"] // 1) * 1

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_zoom[df_zoom["band_m"] == band_m]
    
    for season in ["spring", "autumn"]:
        ss = subset[subset["season"] == season]
        if ss.empty:
            continue
        
        grouped = ss.groupby("sza_bin_1")["snr"].agg(["median","count","std"])
        grouped = grouped[grouped["count"] >= 20]
        
        x = grouped.index.values
        y = grouped["median"].values
        err = grouped["std"].values / np.sqrt(grouped["count"].values)
        
        ax.plot(x, y, color=band_colors[band_m],
                linestyle=season_ls[season],
                linewidth=2, label=f"{season.capitalize()}")
        ax.fill_between(x, y - err, y + err,
                        alpha=0.15, color=band_colors[band_m])
    
    ax.axvline(90, color="black", linewidth=1.5,
               linestyle=":", alpha=0.7, label="Terminator")
    
    # Annotate
    ax.text(0.05, 0.95, "Daylight →", transform=ax.transAxes,
            fontsize=8, color="goldenrod", va="top")
    ax.text(0.65, 0.95, "← Darkness", transform=ax.transAxes,
            fontsize=8, color="navy", va="top")
    
    ax.set_xlabel("Solar Zenith Angle (°)")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Median SNR (dB)")
plt.suptitle("Gray Line Zoom — SZA 82° to 98°, 1° Bins\n"
             "Equinox weeks 2023, DX paths 1k-8k km",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Check 40m spring vs autumn asymmetry in more detail
df_40_eq = df_zoom[df_zoom["band_m"] == 40].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, season in zip(axes, ["spring", "autumn"]):
    ss = df_40_eq[df_40_eq["season"] == season]
    
    # Split by distance bucket
    for dist, color in [("1k-2k","steelblue"), ("2k-4k","goldenrod"), ("4k-8k","tomato")]:
        sub = ss[ss["dist_bucket"] == dist]
        if sub.empty:
            continue
        grouped = sub.groupby("sza_bin_1")["snr"].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 10]
        if grouped.empty:
            continue
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, label=dist)
    
    ax.axvline(90, color="black", linewidth=1.5, linestyle=":", alpha=0.7)
    ax.set_xlabel("Solar Zenith Angle (°)")
    ax.set_ylabel("Median SNR (dB)")
    ax.set_title(f"40m — {season.capitalize()} equinox\nBy distance bucket")
    ax.legend(fontsize=9, title="Distance")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-22, -13)

plt.suptitle("40m Gray Line Asymmetry — Spring vs Autumn Equinox 2023", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compute daylight baseline per band and season
# Baseline = median SNR at SZA 70-85° (well into daylight, away from terminator)

df_eq_dx = df_eq[
    df_eq["dist_bucket"].isin(["1k-2k", "2k-4k", "4k-8k"])
].copy()

df_eq_dx["sza_bin_1"] = (df_eq_dx["mid_sza"] // 1) * 1

# Compute baselines
baselines = {}
for band_m in [10, 20, 40]:
    for season in ["spring", "autumn"]:
        mask = (
            (df_eq_dx["band_m"] == band_m) &
            (df_eq_dx["season"] == season) &
            (df_eq_dx["mid_sza"] >= 70) &
            (df_eq_dx["mid_sza"] <= 85)
        )
        baselines[(band_m, season)] = df_eq_dx[mask]["snr"].median()
        print(f"{band_m}m {season}: baseline = {baselines[(band_m, season)]:.2f} dB")

print("\nBaselines computed.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_eq_dx[
        (df_eq_dx["band_m"] == band_m) &
        (df_eq_dx["mid_sza"] >= 78) &
        (df_eq_dx["mid_sza"] <= 108)
    ]
    
    for season in ["spring", "autumn"]:
        ss = subset[subset["season"] == season]
        if ss.empty:
            continue
        
        baseline = baselines[(band_m, season)]
        
        grouped = ss.groupby("sza_bin_1")["snr"].agg(["median","count","std"])
        grouped = grouped[grouped["count"] >= 20]
        
        x   = grouped.index.values
        y   = grouped["median"].values - baseline
        err = grouped["std"].values / np.sqrt(grouped["count"].values)
        
        ax.plot(x, y,
                color=band_colors[band_m],
                linestyle=season_ls[season],
                linewidth=2.5,
                label=f"{season.capitalize()} (base={baseline:.0f} dB)")
        ax.fill_between(x, y - err, y + err,
                        alpha=0.15, color=band_colors[band_m])

    ax.axvline(90, color="black", linewidth=1.5, linestyle=":",
               alpha=0.8, label="Terminator (SZA=90°)")
    ax.axhline(0, color="gray", linewidth=1, linestyle="--", alpha=0.5)
    ax.axvspan(87, 93, alpha=0.08, color="gold", label="Gray line zone ±3°")

    ax.text(0.05, 0.95, "← Daylight", transform=ax.transAxes,
            fontsize=8, color="goldenrod", va="top")
    ax.text(0.70, 0.95, "Darkness →", transform=ax.transAxes,
            fontsize=8, color="navy", va="top")

    ax.set_xlabel("Solar Zenith Angle (°)")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("SNR anomaly relative to daylight baseline (dB)")
plt.suptitle("Gray Line Transition — Normalized SNR Anomaly\n"
             "Equinox weeks 2023, DX paths 1k-8k km",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("Summer schedule dates:")
for d in fetch_schedule:
    if d.month in [5, 6, 7, 8]:
        print(f"  {d.strftime('%Y-%m-%d')} ({d.strftime('%B')})")

In [ ]:
summer_months = [5, 6, 7, 8]
winter_months = [1, 2]

summer_dates = [d for d in fetch_schedule if d.month in summer_months]
winter_dates = [d for d in fetch_schedule if d.month in winter_months]

print(f"Summer days: {len(summer_dates)}")
print(f"Winter days: {len(winter_dates)}")

def load_40m_days(date_list, label):
    frames = []
    for d in date_list:
        date_str = d.strftime("%Y-%m-%d")
        path = WSPR_DATA_DIR / f"wspr_{date_str}_40m_geo.parquet"
        if not path.exists():
            continue
        df = pd.read_parquet(path)
        df["season_label"] = label
        df["date_str"] = date_str
        df["month"] = d.month
        frames.append(df)
    result = pd.concat(frames, ignore_index=True)
    
    # Add distance bucket
    dist_bins   = [0, 500, 1000, 2000, 4000, 8000, 12000]
    dist_labels = ["<500", "500-1k", "1k-2k", "2k-4k", "4k-8k", "8k+"]
    result["dist_bucket"] = pd.cut(result["distance"], bins=dist_bins, labels=dist_labels)
    print(f"{label}: {len(result):,} rows from {len(frames)} days")
    return result

df_summer_40 = load_40m_days(summer_dates, "summer")
df_winter_40 = load_40m_days(winter_dates, "winter")

In [ ]:
# Focus on daytime DX paths where Es signature should be clearest
# Es on 40m typically supports 500-2500km paths
# Use SZA < 80° to ensure we're well into daylight (sun high overhead)

df_summer_dx = df_summer_40[
    (df_summer_40["path_type"] == "day") &
    (df_summer_40["mid_sza"] < 80) &
    (df_summer_40["dist_bucket"].isin(["500-1k", "1k-2k", "2k-4k"]))
].copy()

df_winter_dx = df_winter_40[
    (df_winter_40["path_type"] == "day") &
    (df_winter_40["mid_sza"] < 80) &
    (df_winter_40["dist_bucket"].isin(["500-1k", "1k-2k", "2k-4k"]))
].copy()

print(f"Summer daytime DX rows: {len(df_summer_dx):,}")
print(f"Winter daytime DX rows: {len(df_winter_dx):,}")

# Plot SNR distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False, sharex=True)

dist_buckets = ["500-1k", "1k-2k", "2k-4k"]
season_colors = {"summer": "tomato", "winter": "steelblue"}

for ax, dist in zip(axes, dist_buckets):
    for label, df, color in [
        ("Summer (May-Aug)", df_summer_dx, "tomato"),
        ("Winter (Jan-Feb)", df_winter_dx, "steelblue")
    ]:
        subset = df[df["dist_bucket"] == dist]["snr"]
        if subset.empty:
            continue
        
        ax.hist(subset, bins=60, range=(-35, 20),
                color=color, alpha=0.5, density=True,
                label=f"{label} (n={len(subset):,})")
        
        # Mark median
        med = subset.median()
        ax.axvline(med, color=color, linewidth=2, linestyle="--", alpha=0.8)
    
    ax.set_xlabel("SNR (dB)")
    ax.set_title(f"Distance: {dist} km")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Density")
plt.suptitle("40m SNR Distribution — Summer vs Winter\n"
             "Daytime paths (SZA < 80°), DX distances",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Define Es threshold — spots above -5 dB on 40m daytime DX 
# are almost certainly Es, not F2
ES_THRESHOLD = -5  # dB

df_summer_500_2k = df_summer_40[
    (df_summer_40["path_type"] == "day") &
    (df_summer_40["mid_sza"] < 80) &
    (df_summer_40["dist_bucket"].isin(["500-1k", "1k-2k"]))
].copy()

# Label each spot
df_summer_500_2k["mode"] = np.where(
    df_summer_500_2k["snr"] >= ES_THRESHOLD, "Es", "F2"
)

# Es fraction by month and hour
df_summer_500_2k["hour"] = df_summer_500_2k["time"].dt.hour

print("Es fraction by month (SZA<80°, 500-2k km, 40m):")
monthly = df_summer_500_2k.groupby("month").apply(
    lambda x: (x["mode"] == "Es").sum() / len(x) * 100
).round(2)
print(monthly.to_string())

print("\nEs fraction by hour:")
hourly = df_summer_500_2k.groupby("hour").apply(
    lambda x: (x["mode"] == "Es").sum() / len(x) * 100
).round(2)
print(hourly.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Split by day vs night
for ax, (time_label, sza_range) in zip(axes, [
    ("Daytime (SZA < 70°)", (0, 70)),
    ("Nighttime (SZA > 100°)", (100, 180))
]):
    subset = df_summer_500_2k[
        (df_summer_500_2k["mid_sza"] >= sza_range[0]) &
        (df_summer_500_2k["mid_sza"] < sza_range[1])
    ]
    
    if subset.empty:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center")
        continue
    
    # SNR distribution by distance bucket
    for dist, color in [("500-1k", "steelblue"), ("1k-2k", "tomato")]:
        s = subset[subset["dist_bucket"] == dist]["snr"]
        if s.empty:
            continue
        ax.hist(s, bins=50, range=(-35, 20),
                color=color, alpha=0.5, density=True,
                label=f"{dist} km (n={len(s):,})")
    
    ax.axvline(ES_THRESHOLD, color="black", linewidth=1.5,
               linestyle="--", label=f"Es threshold ({ES_THRESHOLD} dB)")
    ax.set_xlabel("SNR (dB)")
    ax.set_ylabel("Density")
    ax.set_title(f"Summer 40m — {time_label}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("40m SNR Distribution — Daytime vs Nighttime\n"
             "Summer 2023, 500-2k km paths",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Recompute hourly Es fraction restricted to SZA < 70°
df_es_day = df_summer_500_2k[df_summer_500_2k["mid_sza"] < 70].copy()

print(f"Rows with SZA < 70°: {len(df_es_day):,}")

# Es fraction by hour and month
df_es_day["hour"] = df_es_day["time"].dt.hour
df_es_day["es_flag"] = df_es_day["snr"] >= ES_THRESHOLD

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Es fraction by hour
ax = axes[0]
hourly_es = df_es_day.groupby("hour")["es_flag"].agg(["sum","count"])
hourly_es["es_pct"] = hourly_es["sum"] / hourly_es["count"] * 100

ax.bar(hourly_es.index, hourly_es["es_pct"],
       color="tomato", alpha=0.7, edgecolor="darkred", linewidth=0.5)
ax.set_xlabel("Hour UTC")
ax.set_ylabel("Es fraction (%)")
ax.set_title("Es fraction by hour UTC\n(SZA < 70°, summer 2023)")
ax.set_xticks(range(0, 24, 2))
ax.grid(True, alpha=0.3, axis="y")

# Panel 2: Es fraction by month
ax = axes[1]
monthly_es = df_es_day.groupby("month")["es_flag"].agg(["sum","count"])
monthly_es["es_pct"] = monthly_es["sum"] / monthly_es["count"] * 100
month_names = {5:"May", 6:"Jun", 7:"Jul", 8:"Aug"}

ax.bar(monthly_es.index, monthly_es["es_pct"],
       color="steelblue", alpha=0.7, edgecolor="navy", linewidth=0.5)
ax.set_xlabel("Month")
ax.set_ylabel("Es fraction (%)")
ax.set_title("Es fraction by month\n(SZA < 70°, summer 2023)")
ax.set_xticks(monthly_es.index)
ax.set_xticklabels([month_names[m] for m in monthly_es.index])
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("40m Sporadic-E Activity — Summer 2023\n"
             "500-2k km daytime paths, SNR ≥ -5 dB threshold",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Check sample sizes per hour
hourly_detail = df_es_day.groupby("hour")["es_flag"].agg(["sum","count"])
hourly_detail["es_pct"] = hourly_detail["sum"] / hourly_detail["count"] * 100
hourly_detail.columns = ["es_count", "total_count", "es_pct"]
print("Hourly Es detail:")
print(hourly_detail.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Es fraction by hour — exclude thin hours
ax = axes[0]
valid = hourly_detail[hourly_detail["total_count"] >= 1000]

bars = ax.bar(valid.index, valid["es_pct"],
              color="tomato", alpha=0.7, edgecolor="darkred", linewidth=0.5)

# Color the midday trough differently
for i, (hour, row) in enumerate(valid.iterrows()):
    if 9 <= hour <= 15:
        bars[i].set_facecolor("steelblue")
        bars[i].set_alpha(0.7)

ax.set_xlabel("Hour UTC")
ax.set_ylabel("Es fraction (%)")
ax.set_title("Es fraction by hour UTC\n(SZA < 70°, n ≥ 1000 spots per hour)")
ax.set_xticks(range(0, 24, 2))
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 20)

# Add annotation
ax.annotate("Midday\nsuppression", xy=(12, 9.8), xytext=(14, 16),
            fontsize=8, color="steelblue",
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.2))

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="tomato", alpha=0.7, label="Morning/evening Es"),
    Patch(color="steelblue", alpha=0.7, label="Midday suppression (09-15 UTC)")
], fontsize=8)

# Panel 2: Monthly with error context
ax = axes[1]
monthly_es = df_es_day.groupby("month")["es_flag"].agg(["sum","count"])
monthly_es["es_pct"] = monthly_es["sum"] / monthly_es["count"] * 100
# Bootstrap-style error: binomial SE
monthly_es["se"] = np.sqrt(
    monthly_es["es_pct"]/100 * (1 - monthly_es["es_pct"]/100) / monthly_es["count"]
) * 100

month_names = {5:"May", 6:"Jun", 7:"Jul", 8:"Aug"}
ax.bar(range(len(monthly_es)), monthly_es["es_pct"],
       color="steelblue", alpha=0.7, edgecolor="navy", linewidth=0.5,
       yerr=monthly_es["se"]*3, capsize=5, error_kw={"linewidth":1.5})
ax.set_xlabel("Month")
ax.set_ylabel("Es fraction (%)")
ax.set_title("Es fraction by month\n(SZA < 70°, ±3 SE error bars)")
ax.set_xticks(range(len(monthly_es)))
ax.set_xticklabels([month_names[m] for m in monthly_es.index])
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 16)

# Add spot counts
for i, (month, row) in enumerate(monthly_es.iterrows()):
    ax.text(i, 0.5, f"n={row['count']/1e6:.1f}M",
            ha="center", fontsize=7, color="white", fontweight="bold")

plt.suptitle("40m Sporadic-E Activity — Summer 2023\n"
             "500-2k km daytime paths, SNR ≥ -5 dB threshold",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Reload aggregated data if needed
try:
    _ = df_agg
    print(f"df_agg already loaded: {len(df_agg):,} rows")
except NameError:
    df_agg = pd.read_parquet(PROJECT_ROOT / "data" / "explore_agg_v1.parquet")
    print(f"Loaded df_agg: {len(df_agg):,} rows")

print(f"\nColumns: {list(df_agg.columns)}")
print(f"\nSpot count range: {df_agg['spot_count'].min()} to {df_agg['spot_count'].max():,}")
print(f"SNR median range: {df_agg['snr_median'].min():.1f} to {df_agg['snr_median'].max():.1f} dB")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

band_colors = {10: "tomato", 20: "steelblue", 40: "seagreen"}
band_labels = {10: "10m (28 MHz)", 20: "20m (14 MHz)", 40: "40m (7 MHz)"}

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_agg[
        (df_agg["band_m"] == band_m) &
        (df_agg["spot_count"] >= 10)  # exclude tiny samples
    ].copy()
    
    # Log scale for spot count — spans orders of magnitude
    ax.scatter(subset["snr_median"], np.log10(subset["spot_count"]),
               alpha=0.15, s=8, color=band_colors[band_m])
    
    # Correlation
    corr = subset[["snr_median","spot_count"]].corr().iloc[0,1]
    ax.text(0.05, 0.95, f"r = {corr:.3f}",
            transform=ax.transAxes, fontsize=10,
            va="top", fontweight="bold")
    
    ax.set_xlabel("Median SNR (dB)")
    ax.set_ylabel("log₁₀(spot count)")
    ax.set_title(f"{band_labels[band_m]}")
    ax.grid(True, alpha=0.3)

plt.suptitle("Spot Count vs Median SNR — All Conditions, 2023\n"
             "Each point is one aggregation bin (band × time × distance × path type)",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=False)

sfi_bins = np.arange(110, 350, 20)
sfi_centers = (sfi_bins[:-1] + sfi_bins[1:]) / 2

for col, band_m in enumerate([10, 20, 40]):
    subset = df_agg[
        (df_agg["band_m"] == band_m) &
        (df_agg["path_type"] == "day") &
        (df_agg["spot_count"] >= 10)
    ].copy()
    
    subset["sfi_bin"] = pd.cut(subset["Fobs_mean"], bins=sfi_bins)
    
    # SNR vs SFI
    ax_snr = axes[0, col]
    sfi_snr = subset.groupby("sfi_bin")["snr_median"].agg(["median","std","count"])
    sfi_snr = sfi_snr[sfi_snr["count"] >= 5]
    
    ax_snr.plot(sfi_centers[:len(sfi_snr)], sfi_snr["median"],
                color=band_colors[band_m], linewidth=2)
    ax_snr.fill_between(sfi_centers[:len(sfi_snr)],
                        sfi_snr["median"] - sfi_snr["std"]/2,
                        sfi_snr["median"] + sfi_snr["std"]/2,
                        alpha=0.2, color=band_colors[band_m])
    ax_snr.set_ylabel("Median SNR (dB)")
    ax_snr.set_title(f"{band_labels[band_m]}")
    ax_snr.grid(True, alpha=0.3)
    
    # Spot count vs SFI
    ax_cnt = axes[1, col]
    sfi_cnt = subset.groupby("sfi_bin")["spot_count"].agg(["median","std","count"])
    sfi_cnt = sfi_cnt[sfi_cnt["count"] >= 5]
    
    ax_cnt.plot(sfi_centers[:len(sfi_cnt)], sfi_cnt["median"],
                color=band_colors[band_m], linewidth=2)
    ax_cnt.fill_between(sfi_centers[:len(sfi_cnt)],
                        sfi_cnt["median"] - sfi_cnt["std"]/2,
                        sfi_cnt["median"] + sfi_cnt["std"]/2,
                        alpha=0.2, color=band_colors[band_m])
    ax_cnt.set_ylabel("Median spot count")
    ax_cnt.set_xlabel("F10.7 Solar Flux Index")
    ax_cnt.grid(True, alpha=0.3)

axes[0, 0].set_ylabel("Median SNR (dB)")
axes[1, 0].set_ylabel("Median spot count per bin")
fig.text(0.5, 0.96, "SNR vs Spot Count Response to Solar Flux — Day Paths, 2023",
         ha="center", fontsize=13, fontweight="bold")
fig.text(0.5, 0.92, "Top row: SNR     Bottom row: Spot count",
         ha="center", fontsize=10, color="gray")
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
# Simpler: just look at 10m specifically during high vs low SFI
# restricting to identical conditions — same distance, same path type, same SZA bin

df_10m_sfi = df_agg[
    (df_agg["band_m"] == 10) &
    (df_agg["path_type"] == "day") &
    (df_agg["dist_bucket"].isin(["1k-2k","2k-4k"])) &
    (df_agg["spot_count"] >= 5)
].copy()

df_10m_sfi["sza_bin"] = (df_10m_sfi["mid_sza_med"] // 10) * 10
df_10m_sfi["sfi_cat"] = pd.cut(df_10m_sfi["Fobs_mean"],
                                 bins=[100, 160, 200, 350],
                                 labels=["Low (100-160)","Med (160-200)","High (200+)"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

sfi_colors = {"Low (100-160)": "steelblue", "Med (160-200)": "goldenrod", "High (200+)": "tomato"}

for ax, metric, ylabel in zip(axes,
    ["snr_median", "spot_count"],
    ["Median SNR (dB)", "Median spot count per bin"]):
    
    for sfi_label, color in sfi_colors.items():
        subset = df_10m_sfi[df_10m_sfi["sfi_cat"] == sfi_label]
        if subset.empty:
            continue
        grouped = subset.groupby("sza_bin")[metric].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 3]
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, label=sfi_label)
    
    ax.invert_xaxis()
    ax.set_xlabel("Solar Zenith Angle (°)\n← Overhead     Horizon →")
    ax.set_ylabel(ylabel)
    ax.set_title(f"10m — {ylabel} vs SZA by SFI")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("10m: SNR vs Spot Count Response to SFI\n"
             "Day paths 1k-4k km, 2023", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
df_20m_kp2 = df_agg[
    (df_agg["band_m"] == 20) &
    (df_agg["path_type"] == "day") &
    (df_agg["dist_bucket"].isin(["2k-4k","4k-8k"])) &
    (df_agg["spot_count"] >= 5)
].copy()

df_20m_kp2["sza_bin"] = (df_20m_kp2["mid_sza_med"] // 10) * 10
df_20m_kp2["kp_cat"] = pd.cut(df_20m_kp2["Kp_mean"],
                                bins=[0, 2, 4, 6, 9],
                                labels=["Quiet (0-2)","Unsettled (2-4)",
                                        "Storm (4-6)","Severe (6+)"])

kp_colors = {"Quiet (0-2)":     "steelblue",
             "Unsettled (2-4)": "seagreen",
             "Storm (4-6)":     "goldenrod",
             "Severe (6+)":     "tomato"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, metric, ylabel in zip(axes,
    ["snr_median", "spot_count"],
    ["Median SNR (dB)", "Median spot count per bin"]):

    for kp_label, color in kp_colors.items():
        subset = df_20m_kp2[df_20m_kp2["kp_cat"] == kp_label]
        if subset.empty:
            continue
        grouped = subset.groupby("sza_bin")[metric].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 3]
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, label=kp_label)

    ax.invert_xaxis()
    ax.set_xlabel("Solar Zenith Angle (°)\n← Overhead     Horizon →")
    ax.set_ylabel(ylabel)
    ax.set_title(f"20m DX — {ylabel} vs SZA by Kp")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("20m DX: SNR vs Spot Count Response to Kp\n"
             "Day paths 2k-8k km, 2023", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("Mean SFI by Kp category (20m DX day paths):")
print(df_20m_kp2.groupby("kp_cat")["Fobs_mean"].agg(["mean","std","count"]).round(1))

In [ ]:
# Station count proxy — unique tx_sign per time bin
# We don't have this in df_agg but we can check one high-Kp day vs quiet day

# Find highest Kp day in our dataset
high_kp_days = df_agg[df_agg["Kp_mean"] >= 6]["date"].unique()
low_kp_days  = df_agg[df_agg["Kp_mean"] <= 1]["date"].unique()

print(f"High Kp days (≥6) in dataset: {len(high_kp_days)}")
print(high_kp_days)
print(f"\nLow Kp days (≤1) in dataset: {len(low_kp_days)}")
print(low_kp_days[:5], "...")

In [ ]:
# Compare March 23 (Kp≥6, storm) vs March 20 (quiet, equinox)
# Both are in our dataset — same week, same season

days_to_compare = {
    "2023-03-20 (Quiet)":  "2023-03-20",
    "2023-03-23 (Storm)":  "2023-03-23"
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, date_str in days_to_compare.items():
    # Load 20m geo parquet for this day
    path = WSPR_DATA_DIR / f"wspr_{date_str}_20m_geo.parquet"
    if not path.exists():
        print(f"Missing: {path}")
        continue
    
    df_day = pd.read_parquet(path)
    df_day["time_1h"] = df_day["time"].dt.floor("1h")
    df_day["hour"] = df_day["time"].dt.hour
    
    # Unique transmitters per hour
    tx_per_hour = df_day.groupby("hour")["tx_sign"].nunique()
    # Spot count per hour
    spots_per_hour = df_day.groupby("hour").size()
    # Median SNR per hour
    snr_per_hour = df_day.groupby("hour")["snr"].median()
    
    ls = "-" if "Quiet" in label else "--"
    color = "steelblue" if "Quiet" in label else "tomato"
    
    axes[0].plot(tx_per_hour.index, tx_per_hour.values,
                 color=color, linestyle=ls, linewidth=2, label=label)
    axes[1].plot(snr_per_hour.index, snr_per_hour.values,
                 color=color, linestyle=ls, linewidth=2, label=label)

axes[0].set_xlabel("Hour UTC")
axes[0].set_ylabel("Unique transmitters")
axes[0].set_title("Active TX stations per hour")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(0, 24, 2))

axes[1].set_xlabel("Hour UTC")
axes[1].set_ylabel("Median SNR (dB)")
axes[1].set_title("Median SNR per hour")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(0, 24, 2))

plt.suptitle("20m — Quiet Day vs Storm Day\n"
             "March 20 (Kp≤2) vs March 23 (Kp≥6), same week",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Path loss vs distance — all conditions combined
df_pl = df_agg[df_agg["spot_count"] >= 20].copy()

# Convert dist_bucket to numeric midpoint for plotting
dist_midpoints = {
    "<500":   250,
    "500-1k": 750,
    "1k-2k":  1500,
    "2k-4k":  3000,
    "4k-8k":  6000,
    "8k+":    10000
}
df_pl["dist_mid"] = df_pl["dist_bucket"].map(dist_midpoints)

fig, ax = plt.subplots(figsize=(12, 6))

for band_m in [10, 20, 40]:
    subset = df_pl[df_pl["band_m"] == band_m]
    
    grouped = subset.groupby("dist_mid")["path_loss_med"].agg(
        ["median","std","count"]
    )
    
    x = grouped.index.values
    y = grouped["median"].values
    err = grouped["std"].values
    
    ax.plot(x, y, color=band_colors[band_m], linewidth=2.5,
            marker="o", markersize=6, label=band_labels[band_m])
    ax.fill_between(x, y - err/2, y + err/2,
                    alpha=0.15, color=band_colors[band_m])

# Free-space reference line — 6 dB per doubling of distance
# Anchored at 1000 km
ref_dist = np.array([250, 750, 1500, 3000, 6000, 10000])
ref_loss = 20 * np.log10(ref_dist / 1000) + 38  # anchor at 38 dB at 1000km
ax.plot(ref_dist, ref_loss, "k--", linewidth=1.5, alpha=0.5,
        label="Free-space reference (6 dB/doubling)")

ax.set_xscale("log")
ax.set_xlabel("Distance (km, log scale)")
ax.set_ylabel("Path loss proxy (dB)\n[tx power − SNR]")
ax.set_title("HF Path Loss vs Distance — All Conditions, 2023\n"
             "Higher values = more loss")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which="both")
ax.set_xticks([250, 750, 1500, 3000, 6000, 10000])
ax.set_xticklabels(["250", "750", "1.5k", "3k", "6k", "10k"])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

path_colors = {"day": "goldenrod", "mixed": "purple", "night": "navy"}

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_pl[df_pl["band_m"] == band_m]
    
    for path_type, color in path_colors.items():
        ss = subset[subset["path_type"] == path_type]
        grouped = ss.groupby("dist_mid")["path_loss_med"].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 5]
        if grouped.empty:
            continue
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, marker="o",
                markersize=5, label=path_type.capitalize())
    
    ax.set_xscale("log")
    ax.set_xlabel("Distance (km)")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")
    ax.set_xticks([250, 750, 1500, 3000, 6000, 10000])
    ax.set_xticklabels(["250", "750", "1.5k", "3k", "6k", "10k"])
    ax.set_ylim(30, 65)

axes[0].set_ylabel("Path loss proxy (dB)\n[tx power − SNR]")
plt.suptitle("HF Path Loss vs Distance — Day / Mixed / Night\n"
             "All bands, 2023", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_20m_pl = df_pl[
    (df_pl["band_m"] == 20) &
    (df_pl["path_type"] == "day")
].copy()

df_20m_pl["sfi_cat"] = pd.cut(df_20m_pl["Fobs_mean"],
                               bins=[100, 160, 200, 350],
                               labels=["Low (100-160)", "Med (160-200)", "High (200+)"])

sfi_colors = {"Low (100-160)": "steelblue", 
              "Med (160-200)": "goldenrod", 
              "High (200+)":   "tomato"}

for ax, metric, ylabel in zip(axes,
    ["path_loss_med", "snr_median"],
    ["Path loss proxy (dB)", "Median SNR (dB)"]):
    
    for sfi_label, color in sfi_colors.items():
        subset = df_20m_pl[df_20m_pl["sfi_cat"] == sfi_label]
        grouped = subset.groupby("dist_mid")[metric].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 5]
        if grouped.empty:
            continue
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, marker="o",
                markersize=5, label=sfi_label)
    
    ax.set_xscale("log")
    ax.set_xlabel("Distance (km)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"20m Day — {ylabel} vs Distance by SFI")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which="both")
    ax.set_xticks([250, 750, 1500, 3000, 6000, 10000])
    ax.set_xticklabels(["250", "750", "1.5k", "3k", "6k", "10k"])

plt.suptitle("20m Day Paths — Path Loss and SNR vs Distance by SFI\n2023",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Look at path loss at the longest distances as a function of SFI
# The steepening/collapse at long distances marks the MUF limit

df_long = df_pl[
    (df_pl["path_type"] == "day") &
    (df_pl["dist_bucket"].isin(["2k-4k", "4k-8k", "8k+"]))
].copy()

df_long["sfi_bin"] = pd.cut(df_long["Fobs_mean"],
                             bins=np.arange(110, 320, 20))
df_long["sfi_center"] = df_long["sfi_bin"].apply(
    lambda x: x.mid if pd.notna(x) else np.nan
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

dist_colors = {"2k-4k": "steelblue", "4k-8k": "goldenrod", "8k+": "tomato"}

for ax, band_m in zip(axes, [10, 20, 40]):
    subset = df_long[df_long["band_m"] == band_m]
    
    for dist, color in dist_colors.items():
        ss = subset[subset["dist_bucket"] == dist]
        grouped = ss.groupby("sfi_center")["path_loss_med"].agg(["median","count"])
        grouped = grouped[grouped["count"] >= 3]
        if grouped.empty:
            continue
        ax.plot(grouped.index, grouped["median"],
                color=color, linewidth=2, marker="o",
                markersize=4, label=dist)
    
    ax.set_xlabel("F10.7 Solar Flux Index")
    ax.set_title(f"{band_labels[band_m]}")
    ax.legend(fontsize=9, title="Distance")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Path loss proxy (dB)")
plt.suptitle("Long-Distance Path Loss vs Solar Flux — Day Paths\n"
             "Steepening at high SFI indicates MUF approach",
             fontsize=12)
plt.tight_layout()
plt.show()